# RAMP Demand Model — Saba 2035
**Report:** Pathways to 100% Renewable Electricity on Saba  
**Course:** TU Delft SEN1531 — Design of Integrated Energy Systems (2025/26 Q3)  
**Outputs:** `saba_demand_median_v.csv` · `saba_demand_all_runs_v.csv` · `saba_2035_figures.png`

| Cell | Description | Report section |
|------|-------------|----------------|
| 1 | Version check & imports | — |
| 2 | Diagnostic — inspect add_appliance signature | — |
| 3 | Constants | §2.1–2.2 |
| 4 | User & appliance definitions | §2.3, Table 1 |
| 5 | Monte Carlo simulation (50 runs) | §2.4 Steps 1–3 |
| 6 | Post-processing | §2.4 Steps 4–5 |
| 7 | QA dashboard | §3, Table 2 |
| 8 | CSV export | §6, Table 5 |
| 9 | Figures 1–3 | §4.2–4.4 |


In [1]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore", category=FutureWarning, module="ramp")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from IPython.display import display
import inspect, ramp
from ramp import UseCase, User

print(f"RAMP    : {ramp.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")


RAMP    : 0.5.0
pandas  : 2.3.3
numpy   : 2.3.5


In [2]:
# ── 2. DIAGNOSTIC — run this cell and paste the output here if errors persist
# It prints the exact add_appliance() signature for your RAMP version
# so we can confirm the correct argument order.

print("=== add_appliance signature ===")
print(inspect.signature(User.add_appliance))
print()

# Minimal smoke-test: one user, one appliance, one simulation day
_u = User(user_name="test", num_users=1)
_a = _u.add_appliance("TestFridge", number=1, power=60,
                       num_windows=1, func_time=1440,
                       time_fraction_random_variability=0, flat=1)
_a.windows(window_1=[0, 1440])

# Confirm power stored as numeric
print(f"power stored as : {type(_u.App_list[0].power)}  value={_u.App_list[0].power}")
print()
if isinstance(_u.App_list[0].power, str):
    print("⚠️  power is a STRING — check the signature output above.")
    print("   The argument order in your RAMP version may differ from what we assumed.")
else:
    print("✓  power is numeric — Cell 4 will work correctly.")


=== add_appliance signature ===
(self, *args, **kwargs)

power stored as : <class 'numpy.ndarray'>  value=[60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 60. 60. 60. 60. 6

In [3]:
# ── 3. CONSTANTS  (report §2.1–2.2) ───────────────────────────────────────
DATE_START, DATE_END = "2035-01-01", "2035-12-31"
N_RUNS               = 50
TARGET_MWH           = 9_000
W_TO_MW              = 1e-6
TOURISM_SHARE        = 0.18
TOURISM_PEAK_OCC     = 0.85
TOURISM_MONTHLY      = {1:.85, 2:.85, 3:.85,  4:.85,
                        5:.40, 6:.40, 7:.40,  8:.40,
                        9:.40, 10:.40, 11:.85, 12:.85}
print("✓  Constants set")


✓  Constants set


In [4]:
# ── 4. USER & APPLIANCE DEFINITIONS  (report §2.3, Table 1) ───────────────
# Every argument to add_appliance() is an explicit keyword.
# power, number, func_time are explicitly cast to native Python types
# to prevent RAMP from storing them as numpy dtypes or strings.

def add(user, name, number, power, windows, func_time,
        variability=0.0, flat=0, occasional_use=1.0):
    """Create one appliance and set its usage windows."""
    app = user.add_appliance(
        name                            = str(name),
        number                          = int(number),
        power                           = int(power),       # int avoids dtype='<U6' bug
        num_windows                     = int(len(windows)),
        func_time                       = int(func_time),
        time_fraction_random_variability= float(variability),
        flat                            = int(flat),
        occasional_use                  = float(occasional_use),
    )
    app.windows(**{f"window_{i+1}": w for i, w in enumerate(windows)})

# ── Basic Residential — 500 hh  (100% LED, no AC — EEP 2024-25) ───────────
hA = User(user_name="Basic Residential", num_users=500)
add(hA, "Fridge",        1,  60, [[0,1440]],              1440, flat=1)
add(hA, "LED Lighting",  4,  10, [[360,540],[1080,1320]],  240, .2)
add(hA, "TV",            1,  80, [[1080,1320],[360,480]],  180, .2)
add(hA, "Ceiling Fan",   2,  50, [[1200,1440],[0,360]],    480, .2)
add(hA, "Phone Charger", 2,  10, [[1200,1440]],            120, .3)

# ── Higher Residential — 200 hh  (~30% AC, occasional_use=0.45 — QA Table 4)
hB = User(user_name="Higher Residential", num_users=200)
add(hB, "Fridge",          1,   60, [[0,1440]],              1440, flat=1)
add(hB, "LED Lighting",    5,   10, [[360,540],[1080,1320]],  240, .2)
add(hB, "TV",              1,   80, [[1080,1320],[360,480]],  180, .2)
add(hB, "Air Conditioner", 1, 1200, [[840,1080],[1320,1440]], 360, .15, occasional_use=0.45)
add(hB, "Washer/Dryer",    1, 2000, [[480,720]],               60, .2,  occasional_use=0.30)

# ── Tourism Guestrooms — 70 rooms ──────────────────────────────────────────
to = User(user_name="Tourism Guestroom", num_users=70)
add(to, "AC",              1, 1200, [[720,1080],[1320,1440]], 480, .15, occasional_use=0.45)
add(to, "Hot Water Boiler",1, 2000, [[360,540]],               60, .2)
add(to, "LED Lighting",    4,   10, [[360,540],[1080,1320]],  240, .2)
add(to, "TV",              1,   80, [[1080,1320]],            180, .2)

# ── Commercial — 30 units ──────────────────────────────────────────────────
co = User(user_name="Commercial", num_users=30)
add(co, "Commercial AC",   1, 2500, [[480,1080]],  480, .1,  flat=1)
add(co, "LED Lighting",   10,   10, [[420,1080]],  600, .1,  flat=1)
add(co, "Refrigeration",   1,  400, [[0,1440]],   1440,       flat=1)
add(co, "Dive Compressor", 1, 3000, [[480,720]],   120, .2,  occasional_use=0.40)

# ── EV Charging — 23 EVs, window 19:00–24:00 (QA Table 4) ─────────────────
ev = User(user_name="EV Charging", num_users=23)
add(ev, "EV Charger",      1, 7400, [[1140,1440]], 180, .2, occasional_use=0.70)

# ── Public Lighting — 925 × 40 W LED, dusk-to-dawn ────────────────────────
pu = User(user_name="Public Lighting", num_users=1)
add(pu, "Street Lighting", 925, 40, [[1080,1440],[0,360]], 720, flat=1)

# Verify power is numeric before proceeding
for u in [hA, hB, to, co, ev, pu]:
    for app in u.App_list:
        assert not isinstance(app.power, str), \
            f"power is a string for {app.name} — check add_appliance signature"
print("✓  6 user categories defined — all power values are numeric")


✓  6 user categories defined — all power values are numeric


In [ ]:
# ── 5. SIMULATION — 50 Monte Carlo runs  (report §2.4 Steps 2–3) ──────────
uc      = UseCase(users=[hA,hB,to,co,ev,pu], date_start=DATE_START, date_end=DATE_END)
idx_min = pd.date_range(start=DATE_START, periods=365*1440, freq="1min")

all_runs = {}
for i in range(N_RUNS):
    flat = np.concatenate(uc.generate_daily_load_profiles())
    assert len(flat) == len(idx_min), f"Run {i+1}: length mismatch"
    all_runs[f"run_{i+1:02d}"] = flat
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{N_RUNS} runs complete")

print(f"✓  Simulation complete — {N_RUNS} runs × {365*1440:,} minutes")


You will simulate 365 day(s) from 2035-01-01 00:00:00 until 2036-01-01 00:00:00


In [ ]:
# ── 6. POST-PROCESSING  (report §2.4 Steps 4–5) ───────────────────────────
df_h = pd.DataFrame(all_runs, index=idx_min).resample("1h").mean() * W_TO_MW

for m, occ in TOURISM_MONTHLY.items():
    df_h.loc[df_h.index.month == m] *= (
        (1 - TOURISM_SHARE) + TOURISM_SHARE * (occ / TOURISM_PEAK_OCC)
    )

median_raw    = df_h.median(axis=1)
scale         = TARGET_MWH / median_raw.sum()
median_scaled = median_raw * scale
df_h_scaled   = df_h       * scale
p10           = df_h_scaled.quantile(0.10, axis=1)
p90           = df_h_scaled.quantile(0.90, axis=1)

print(f"✓  Scale factor  : {scale:.4f}  (expected ≈ 4.32)")
print(f"   Annual energy : {median_scaled.sum():,.0f} MWh  [target {TARGET_MWH:,}]")
print(f"   P10 / P90     : {p10.sum():,.0f} / {p90.sum():,.0f} MWh")
print(f"   Peak demand   : {median_scaled.max():.3f} MW")


In [ ]:
# ── 7. QA DASHBOARD  (report §3, Table 2) ─────────────────────────────────
peak  = median_scaled.max()
lf    = median_scaled.mean() / peak
ratio = peak / median_scaled.between_time("02:00", "05:00").mean()
nmin  = median_scaled.between_time("22:00", "05:00").min()

def status(condition): return "✅  PASS" if condition else "❌  FAIL"

display(pd.DataFrame({
    "Simulated":     [f"{median_scaled.sum():,.0f}", f"{peak:.3f}",
                      f"{median_scaled.min():.3f}",  f"{lf:.3f}",
                      f"{ratio:.1f}×",               f"{nmin:.3f}"],
    "Expected range":["8 500–10 500","1.5–2.5","> 0.25","0.25–0.45","< 8×","> 0.30"],
    "Source":        ["pv-magazine 2019; RMI/SEC 2023","Phase 3 grid sizing",
                      "Baseload: fridges + lighting","Caribbean island benchmarks",
                      "Engineering minimum","Fridges + streetlights overnight"],
    "Status":        [status(8500 <= median_scaled.sum() <= 10500),
                      status(1.5  <= peak                <= 2.5),
                      status(median_scaled.min()          > 0.25),
                      status(0.25 <= lf                  <= 0.45),
                      status(ratio                        < 8),
                      status(nmin                         > 0.30)],
}, index=["Annual energy (MWh)","Peak demand (MW)","Min demand (MW)",
          "Load factor","Peak/off-peak ratio","Night min 22–05 (MW)"]))

if peak > 2.5:
    print("\n⚠️  Peak > 2.5 MW — apply ±20% sensitivity band in Calliope (report §5, Table 4)")


In [ ]:
# ── 8. CSV EXPORT  (report §6, Table 5) ───────────────────────────────────
out = median_scaled.rename("demand_MW")
out.index.name = "utc_timestamp"
out.to_csv("saba_demand_median_v.csv", float_format="%.6f")

all_out = df_h_scaled.copy()
all_out.index.name = "utc_timestamp"
all_out.columns    = [f"run_{i+1:02d}" for i in range(N_RUNS)]
all_out["median"]  = median_scaled
all_out["p10"]     = p10
all_out["p90"]     = p90
all_out.to_csv("saba_demand_all_runs_v.csv", float_format="%.6f")

print("✓  saba_demand_median_v.csv    — 8,760 rows × 1 col  (Calliope input)")
print(f"✓  saba_demand_all_runs_v.csv  — 8,760 rows × {N_RUNS+3} cols (sensitivity)")


In [ ]:
# ── 9. FIGURES 1–3  (report §4.2–4.4) ────────────────────────────────────
%matplotlib inline
BLUE, ORANGE = "#2171b5", "#d95f02"
fig, axes = plt.subplots(3, 1, figsize=(13, 12))
fig.suptitle("Saba 2035 — RAMP Electricity Demand Model", fontsize=12, fontweight="bold")

ax = axes[0]
ax.fill_between(median_scaled.index, p10, p90, alpha=0.18, color=BLUE, label="P10–P90 band (50 runs)")
ax.plot(median_scaled.index, median_scaled, lw=0.7, color=BLUE, label="Median")
ax.axhline(peak, color=ORANGE, lw=1.2, ls="--", label=f"Peak  {peak:.2f} MW")
ax.set(title="Figure 1 — Annual demand profile", ylabel="MW")
ax.legend(fontsize=9); ax.grid(alpha=0.2); ax.margins(x=0)

ax = axes[1]
hmed = median_scaled.groupby(median_scaled.index.hour).mean()
hp10 = p10.groupby(p10.index.hour).mean()
hp90 = p90.groupby(p90.index.hour).mean()
ax.fill_between(hp10.index, hp10, hp90, alpha=0.18, color=BLUE, label="P10–P90")
ax.plot(hmed.index, hmed, color=BLUE, lw=1.8, marker="o", ms=4, label="Median")
ax.axvspan(6,  9,  alpha=0.07, color="#fc8d62", label="Morning peak 06–09")
ax.axvspan(18, 22, alpha=0.07, color=ORANGE,    label="Evening peak 18–22")
ax.set(title="Figure 2 — Average daily load shape", xlabel="Hour of day", ylabel="MW")
ax.set_xticks(range(24)); ax.set_xlim(-0.5, 23.5)
ax.legend(fontsize=9); ax.grid(alpha=0.2, axis="y")

ax = axes[2]
mm   = median_scaled.resample("ME").sum()
mp10 = p10.resample("ME").sum()
mp90 = p90.resample("ME").sum()
colors = ["#2171b5" if m in [11,12,1,2,3,4] else "#74a9cf" for m in mm.index.month]
x = range(len(mm))
ax.bar(x, mm.values, color=colors, alpha=0.8, zorder=2)
ax.errorbar(x, mm.values, yerr=[mm.values-mp10.values, mp90.values-mm.values],
            fmt="none", color="#08519c", capsize=5, lw=1.4, zorder=3)
ax.set(title="Figure 3 — Monthly demand & tourism seasonality", ylabel="MWh")
ax.set_xticks(x); ax.set_xticklabels(mm.index.strftime("%b"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.grid(alpha=0.2, axis="y", zorder=0)
ax.legend(handles=[Patch(fc="#2171b5", alpha=.8, label="High season (Nov–Apr)"),
                   Patch(fc="#74a9cf", alpha=.8, label="Low season  (May–Oct)")], fontsize=9)

plt.tight_layout()
plt.savefig("saba_2035_figures.png", dpi=150, bbox_inches="tight")
print("✓  saba_2035_figures.png")
plt.show()
